# Simple RAG — Educational Question-Answering System

















In [12]:
!pip install pypdf -q
!pip install openai chromadb -q

from pypdf import PdfReader

reader = PdfReader("/Python_book.pdf")  # fayl nomini o'zingiznikiga almashtiring

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print(f"Jami {len(text)} belgi chiqarildi")
print(text[:500])  # birinchi 500 belgini ko'rib tekshiramiz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 786.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.9/178.9 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.9/61.9 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 716.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

In [8]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

In [9]:
with open("python_kitob.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("Fayl saqlandi: python_kitob.txt")

Fayl saqlandi: python_kitob.txt


In [10]:
def split_into_chunks(text, chunk_size=500, overlap=80):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

chunks = split_into_chunks(text)
print(f"Jami {len(chunks)} ta bo'lak hosil bo'ldi")
print("Birinchi bo'lak namunasi:\n")
print(chunks[0])

Jami 368 ta bo'lak hosil bo'ldi
Birinchi bo'lak namunasi:

     PYTHON ASOSLARI | ABBOSBEK IBRAGIMOV  
 
 
1 
 
 
Python  asoslari 
 
 
     PYTHON ASOSLARI | ABBOSBEK IBRAGIMOV  
 
 
2 
MUNDARIJA 
 
I - BOB. Python dasturlash tili va sintaksisi 
1.1.Python dasturlash tili va uning imkoniyatlari…….…………………………  3 
1.2. Python dasturlash tili sintaksisi.....………………………………………....   5                        
1.3. Pythonda o’zgaruvchilar...………………………………………………....   8 
1.4. Python operatorlari......… …….………………………………………..…   12                                     


In [13]:
import chromadb
from openai import OpenAI

client_ai = OpenAI()  # API kalitni environment'dan avtomatik oladi
chroma_client = chromadb.Client()

# Kolleksiya (bazadagi "jadval") yaratamiz
collection = chroma_client.create_collection(name="python_kitob")

# Embedding olish funksiyasi
def get_embedding(text):
    response = client_ai.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

# Har bir bo'lakni vektorlashtirib bazaga qo'shamiz
for i, chunk in enumerate(chunks):
    embedding = get_embedding(chunk)
    collection.add(
        ids=[str(i)],
        embeddings=[embedding],
        documents=[chunk],
        metadatas=[{"source": "python_kitob.pdf", "chunk_index": i}]
    )
    if i % 50 == 0:
        print(f"{i}/{len(chunks)} bo'lak qayta ishlandi...")

print("Barcha bo'laklar bazaga saqlandi!")

0/368 bo'lak qayta ishlandi...
50/368 bo'lak qayta ishlandi...
100/368 bo'lak qayta ishlandi...
150/368 bo'lak qayta ishlandi...
200/368 bo'lak qayta ishlandi...
250/368 bo'lak qayta ishlandi...
300/368 bo'lak qayta ishlandi...
350/368 bo'lak qayta ishlandi...
Barcha bo'laklar bazaga saqlandi!


In [17]:
def ask_question(question, top_k=6):
    # 1. Savolni vektorlashtirish
    question_embedding = get_embedding(question)

    # 2. Bazadan eng mos bo'laklarni topish
    results = collection.query(
        query_embeddings=[question_embedding],
        n_results=top_k
    )

    retrieved_chunks = results["documents"][0]
    sources = results["metadatas"][0]

    # 3. Kontekstni yig'ish
    context = "\n\n---\n\n".join(retrieved_chunks)

    # 4. Promptni tuzish
    prompt = f"""Quyidagi kontekst asosida savolga javob bering.
Agar javob kontekstda bo'lmasa, "Bu ma'lumot hujjatda mavjud emas" deb javob bering.
O'zingizdan hech narsa qo'shmang, faqat berilgan matn asosida javob bering.

KONTEKST:
{context}

SAVOL: {question}

JAVOB:"""

    # 5. Modelni chaqirish
    response = client_ai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    answer = response.choices[0].message.content

    # 6. Natijani chiqarish
    print("JAVOB:")
    print(answer)
    print("\nMANBALAR:")
    for i, chunk in enumerate(retrieved_chunks):
        print(f"[{i+1}] ...{chunk[:100]}...")

    return answer

# Test qilib ko'ramiz
ask_question("Python dasturlash tilining o'zgaruvchilari haqida gapiring")

JAVOB:
Bu ma'lumot hujjatda mavjud emas.

MANBALAR:
[1] ...")  
print ("Dasturlashni o'rganamiz") 
 
Izohlar dastur kodini o'qiyotganlar uchun foydali bo'ladi ...
[2] ...uqori ekanligini anglatadi. Python nafaqat web 
sohasida balki sun’iy intellekt va robotexnika sohas...
[3] ...ida qo`llanishidan farq qiladi. 
Python da for operatori biroz murakkabroq, lekin while sikliga qara...
[4] ...yozilgan . Python dasturida ishlaydigan 
foydalanuvchilar uchun  uning sintaksisi, asosiy operatorl ...
[5] ... invertlaydi (teskarisiga o’zgartiradi) 
 <<   -  O’ngdan chapga nollarni siljitib, chapdagi chetki...
[6] ...ida  #  belgisi ishlatiladi. Ammo 3 talik qo’shtirnoq ichiga yozilgan matnni 
o’zgaruvchiga biriktir...


"Bu ma'lumot hujjatda mavjud emas."

In [16]:
ask_question("Pythonda o'zgaruvchi qanday e'lon qilinadi va qanday nomlanishi kerak?", top_k=8)

JAVOB:
Pythonda o'zgaruvchi qiymatni tenglashtirish orqali e'lon qilinadi. O'zgaruvchi nomi harf yoki tag chiziq bilan boshlanishi kerak, raqam bilan boshlanmasligi, faqat harflar, raqamlar va tag chiziqdan iborat bo'lishi mumkin. O'zgaruvchi nomlari katta-kichik harflarni farqlaydi va orasida bo'shliq (probel) bo'lmasligi kerak.

MANBALAR:
[1] ...ldindan ma'lum bir 
qiymat beriladi va bu qiymatni o'zgartirib bo'lmaydi. 
 
Pythonda o’zgaruvchilar...
[2] ... (probel) bo’lmasligi kerak; 
# To'g'ri nomlangan o'zgaruvchilar: 
 
myvar  = "Python" 
my_var = "Py...
[3] ... bilan ifodalanishi mumkin. Ularni ifodalash uchun ayrim  
qoidalar mavjud: 
 O’zgaruvchi nomi harf...
[4] ...sh 
tillaridan farqli, Python o’zgaruvchilarni e’lon qilish uchun alohida buyruqqa ega emas. 
O’zgar...
[5] ...
92 
Pythonda Istisnolar bilan ishlash 
Agar kodimizda xatolik yuz bersa yoki istisno holatlar bo’li...
[6] ... mavjud. Masalan: agarda bitta tasodifiy raqamlar ketma -ketligidan ko`p marta 
foydalanishga e

"Pythonda o'zgaruvchi qiymatni tenglashtirish orqali e'lon qilinadi. O'zgaruvchi nomi harf yoki tag chiziq bilan boshlanishi kerak, raqam bilan boshlanmasligi, faqat harflar, raqamlar va tag chiziqdan iborat bo'lishi mumkin. O'zgaruvchi nomlari katta-kichik harflarni farqlaydi va orasida bo'shliq (probel) bo'lmasligi kerak."

In [18]:
ask_question("Python tsikllari (for va while) haqida ayting", top_k=6)

JAVOB:
Python dasturlash tilida ikki xil sikl ishlatiladi: while va for sikllari. While sikliga odatda bir shart berish kerak bo’ladi va o’sha shart bajarilmaguncha u ko’rsatgan amalni qayta-qayta bajaraveradi. For sikli esa obyektlar ketma-ketligida iteratsiyani amalga oshiradi va har bir o’tish vaqtida sikl tanasini bajaradi. For sikli asosan to’plam va ro’yxatlar bilan ishlatiladi. While sikli shart rost bo’lganda bajariladi, shart yolg’on bo’lganda esa sikl tugatiladi. For sikli esa iteratsiya qilinadigan obyekt bo’ylab o’tadi va har bir elementga amal bajaradi.

MANBALAR:
[1] ...
 
 
     PYTHON ASOSLARI | ABBOSBEK IBRAGIMOV  
 
 
61 
Pythonda sikllar 
Python dasturlash tilida ...
[2] ...ida qo`llanishidan farq qiladi. 
Python da for operatori biroz murakkabroq, lekin while sikliga qara...
[3] ...iz. 
 
 
while sikli 
while sikliga odatda  bir shart berish kerak bo’ladi va o’sha shart bajarilmag...
[4] ...hib ketaversin. Natijada o’zgaruvchimiz toki 10 ga yetguncha ushbu amalni 
b

'Python dasturlash tilida ikki xil sikl ishlatiladi: while va for sikllari. While sikliga odatda bir shart berish kerak bo’ladi va o’sha shart bajarilmaguncha u ko’rsatgan amalni qayta-qayta bajaraveradi. For sikli esa obyektlar ketma-ketligida iteratsiyani amalga oshiradi va har bir o’tish vaqtida sikl tanasini bajaradi. For sikli asosan to’plam va ro’yxatlar bilan ishlatiladi. While sikli shart rost bo’lganda bajariladi, shart yolg’on bo’lganda esa sikl tugatiladi. For sikli esa iteratsiya qilinadigan obyekt bo’ylab o’tadi va har bir elementga amal bajaradi.'

In [20]:
ask_question("Python qachon yaratilgan va uni kim ixtiro qilgan?", top_k=6)

JAVOB:
Bu ma'lumot hujjatda mavjud emas.

MANBALAR:
[1] ...iga kiradi. Python yuqori darajadagi 
ma'lumotlar strukturasi va oddiy lekin samarador obyektga yo'n...
[2] ...ida qo`llanishidan farq qiladi. 
Python da for operatori biroz murakkabroq, lekin while sikliga qara...
[3] ...yaxshiroq modullik 
va kodni yuqori darajada qayta ishlatilishini ta'minlaydi. 
Siz allaqachon bilga...
[4] ...uqori ekanligini anglatadi. Python nafaqat web 
sohasida balki sun’iy intellekt va robotexnika sohas...
[5] ...qilamizki, agar 
x o’zgaruvchi mavjud bo’lmasa, bu haqida xabar berilsin: 
try: 
    print(x) 
excep...
[6] ...yozilgan . Python dasturida ishlaydigan 
foydalanuvchilar uchun  uning sintaksisi, asosiy operatorl ...


"Bu ma'lumot hujjatda mavjud emas."

In [23]:
ask_question("Python dasturlash tilida sikllar haqida ayting", top_k=6)

JAVOB:
Python dasturlash tilida sikllar "for" va "while" operatorlari yordamida amalga oshiriladi. For operatori obyektlar ketma-ketligida iteratsiyani bajaradi va har bir o’tish vaqtida sikl tanasini bajaradi. For sikli asosan to’plam va ro’yxatlar bilan ishlatiladi. While sikli esa shart bajarilganda davom etadi. Python dasturlash tilida siklni to’xtatish uchun "break" kalit so’zi ishlatiladi, bu siklni to’xtatadi, hattoki sikl to’xtamagan bo’lsa ham.

MANBALAR:
[1] ...uqori ekanligini anglatadi. Python nafaqat web 
sohasida balki sun’iy intellekt va robotexnika sohas...
[2] ...")  
print ("Dasturlashni o'rganamiz") 
 
Izohlar dastur kodini o'qiyotganlar uchun foydali bo'ladi ...
[3] ...ida qo`llanishidan farq qiladi. 
Python da for operatori biroz murakkabroq, lekin while sikliga qara...
[4] ...bajarish haqida 
ma’lumotlar keltirilib miso llar yordamida tushuntirilgan. Shuningdek, 
modul tushu...
[5] ...r tushunchasi……..…………………………………..  102 
6.3. Sinflarda vorislik tushunchasi…..…….

'Python dasturlash tilida sikllar "for" va "while" operatorlari yordamida amalga oshiriladi. For operatori obyektlar ketma-ketligida iteratsiyani bajaradi va har bir o’tish vaqtida sikl tanasini bajaradi. For sikli asosan to’plam va ro’yxatlar bilan ishlatiladi. While sikli esa shart bajarilganda davom etadi. Python dasturlash tilida siklni to’xtatish uchun "break" kalit so’zi ishlatiladi, bu siklni to’xtatadi, hattoki sikl to’xtamagan bo’lsa ham.'